# AIMLCZG523 Machine Learning Operations (MLOps) - Assignment 01
## End-to-End ML Development, CI/CD, Containerization & Production Deployment

**Student Name:** Ram Narayan  
**Student ID / Roll:** 2025af05183  
**Email:** 2025af05183@wilp.bits-pilani.ac.in  
**Institution:** BITS Pilani (Work Integrated Learning Programmes)  
**Dataset:** UCI Heart Disease (Cleveland Clinic Foundation)  

---

### 1. Project Overview & Environment Initialization
This notebook demonstrates:
1. Automated acquisition and cleaning of the UCI Heart Disease dataset.
2. Exploratory Data Analysis (EDA) with statistical profiling and clinical visualizations.
3. Feature engineering and preprocessing pipelines using `ColumnTransformer` (scaling, encoding, imputation).
4. Multi-model classification (Logistic Regression, Random Forest, Gradient Boosting) with 5-Fold Stratified Cross-Validation.
5. MLflow experiment tracking and model serialization.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Ensure project root is in python path
sys.path.append('..')
from src.data.download_data import download_dataset
from src.data.preprocess import prepare_datasets, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, ALL_FEATURES
from src.models.predict import get_predictor

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Environment initialized successfully.')

### 2. Data Acquisition & Schema Inspection
We fetch the official UCI Cleveland Heart Disease dataset containing 14 clinical features and target angiographic status.

In [2]:
raw_df = download_dataset('../data/raw/heart_disease_raw.csv')
print(f'Total records: {len(raw_df)}')
raw_df.head()

### 3. Missing Value & Target Encoding Analysis
In accordance with clinical MLOps standards, angiographic disease status `num` (values 0-4) is mapped to binary risk:  
- `target = 0`: Absence of Heart Disease (< 50% arterial diameter narrowing)  
- `target = 1`: Presence of Heart Disease (>= 50% arterial diameter narrowing)  

In [3]:
raw_df['target'] = (raw_df['num'] > 0).astype(int)
print('Target Distribution:')
print(raw_df['target'].value_counts(normalize=True).round(3))
print('\nMissing values per column:')
print(raw_df.isnull().sum()[raw_df.isnull().sum() > 0])

### 4. Exploratory Data Analysis (EDA) Visualizations
Examining distributions of continuous biomarkers: Age, Resting Blood Pressure (`trestbps`), Serum Cholesterol (`chol`), Maximum Heart Rate (`thalach`), and ST Depression (`oldpeak`).

In [4]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(data=raw_df, x='age', hue='target', kde=True, ax=axes[0], palette=['#2b5c8f', '#d95f02'])
axes[0].set_title('Age Distribution by Target')

sns.histplot(data=raw_df, x='chol', hue='target', kde=True, ax=axes[1], palette=['#2b5c8f', '#d95f02'])
axes[1].set_title('Serum Cholesterol (mg/dl)')

sns.histplot(data=raw_df, x='thalach', hue='target', kde=True, ax=axes[2], palette=['#2b5c8f', '#d95f02'])
axes[2].set_title('Max Heart Rate (thalach)')
plt.tight_layout()
plt.show()

### 5. Feature Engineering and Stratified Splitting
We construct a production `ColumnTransformer` with `SimpleImputer` and `StandardScaler` for continuous features, and `OneHotEncoder` for nominal attributes, ensuring zero data leakage.

In [5]:
X_train, X_test, y_train, y_test, preprocessor = prepare_datasets('../data/raw/heart_disease_raw.csv')
print(f'Train samples: {len(X_train)}, Test samples: {len(X_test)}')
print(f'Train positive class ratio: {y_train.mean():.3f}, Test positive ratio: {y_test.mean():.3f}')

### 6. Production Model Inference Verification
Testing the champion serialized model pipeline with sample clinical payloads.

In [6]:
predictor = get_predictor()

healthy_patient = {
    'age': 45.0, 'sex': 0.0, 'cp': 2.0, 'trestbps': 118.0, 'chol': 210.0,
    'fbs': 0.0, 'restecg': 0.0, 'thalach': 172.0, 'exang': 0.0,
    'oldpeak': 0.0, 'slope': 1.0, 'ca': 0.0, 'thal': 3.0
}

high_risk_patient = {
    'age': 67.0, 'sex': 1.0, 'cp': 4.0, 'trestbps': 160.0, 'chol': 286.0,
    'fbs': 0.0, 'restecg': 2.0, 'thalach': 108.0, 'exang': 1.0,
    'oldpeak': 1.5, 'slope': 2.0, 'ca': 3.0, 'thal': 3.0
}

res_healthy = predictor.predict_single(healthy_patient)
res_high_risk = predictor.predict_single(high_risk_patient)

print('Healthy Profile Prediction:', res_healthy)
print('High-Risk Profile Prediction:', res_high_risk)